In [14]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import tensorflow as tf
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from keras.layers import Input, Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from keras import Sequential




# Select the CPU and TensorFlow's backend.
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1" #disabilita GPU
os.environ["KERAS_BACKEND"] = "tensorflow"
# os.environ["KERAS_BACKEND"] = "jax"
# os.environ["KERAS_BACKEND"] = "torch"

import keras
print(keras.__version__)
# Fixed random seed for repeatability.

seed = 42
keras.utils.set_random_seed(seed)
np.random.seed(seed)

import warnings
warnings.filterwarnings("ignore")


3.12.0


In [27]:
import tensorflow as tf
import pandas as pd
import os

# --- Configurazioni ---
IMG_SIZE = 128       # dimensione a cui ridimensionare le immagini
N_CLASSES = 81       # numero totale di classi
BATCH_SIZE = 32
BASE_DIR = "./GroceryStoreDataset/dataset" # cartella dove ci sono train/, val/, test/

# --- Funzione per leggere un txt e creare un dataframe ---
def read_txt(file_path):
    df = pd.read_csv(
        file_path,
        header=None,
        names=["path", "label", "extra"],
        sep=","
    )
    # Prepend BASE_DIR ai percorsi
    df['path'] = df['path'].apply(lambda x: os.path.join(BASE_DIR, x))
    return df

# --- Carica train, val, test ---
train_df = read_txt(os.path.join(BASE_DIR, "train.txt"))
val_df   = read_txt(os.path.join(BASE_DIR, "val.txt"))
test_df  = read_txt(os.path.join(BASE_DIR, "test.txt"))

# --- Funzione per caricare e preprocessare le immagini ---
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    label = tf.one_hot(label, depth=N_CLASSES)
    return image, label

# --- Dataset TensorFlow ottimizzato ---
def create_dataset(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df['label'].values))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.cache()  # memorizza in RAM, velocizza le epoche
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, shuffle=True)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)

print("Dataset pronto!")
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))


Dataset pronto!
Train: 2640 Val: 296 Test: 2485


In [29]:

def run_experiment(train_ds, val_ds, test_ds,
                           filters, rate, lr, epochs):

    # Definiamo l'input shape
    input_shape = (IMG_SIZE, IMG_SIZE, 3)

    model = Sequential([
        Input(shape=input_shape),

        Conv2D(filters=filters, kernel_size=(3,3), activation='relu'),
        MaxPooling2D((2,2)),

        Conv2D(filters=filters**2, kernel_size=(3,3), activation='relu'),
        MaxPooling2D((2,2)),

        Conv2D(filters=filters**4, kernel_size=(3,3), activation='relu'),
        MaxPooling2D((2,2)),

        Flatten(),
        Dropout(rate=rate),
        Dense(N_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(train_ds, validation_data=val_ds, epochs=epochs)
    test_loss, test_acc = model.evaluate(test_ds)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")


 

In [30]:
run_experiment(train_ds, val_ds, test_ds,
                       filters=16, rate=0.15, lr=1e-3, epochs=5)


Epoch 1/5


KeyboardInterrupt: 